# Data Loading

In [2]:
from pathlib import Path

DATA_DIR = Path("../data")
TRAIN_PATH = DATA_DIR / "raw/training_set_VU_DM.csv"
TEST_PATH = DATA_DIR / "raw/test_set_VU_DM.csv"

print(TRAIN_PATH.exists(), TEST_PATH.exists())

True True


In [3]:
from collections.abc import Hashable
from pandas.api.typing.aliases import DtypeArg

DTYPES: dict[Hashable, DtypeArg] = {
    # IDs / categorical integer IDs
    "srch_id": "int32",
    "site_id": "int16",
    "visitor_location_country_id": "int16",
    "prop_country_id": "int16",
    "prop_id": "int32",
    "srch_destination_id": "int32",
    # visitor history
    # nullable floats
    "visitor_hist_starrating": "float32",
    "visitor_hist_adr_usd": "float32",
    # property features
    "prop_starrating": "int8",
    "prop_review_score": "float32",
    "prop_brand_bool": "int8",
    "prop_location_score1": "float32",
    "prop_location_score2": "float32",
    "prop_log_historical_price": "float32",
    # train-only display / target-related columns
    "position": "int16",
    "gross_bookings_usd": "float32",
    "click_bool": "int8",
    "booking_bool": "int8",
    # price / promotion
    "price_usd": "float32",
    "promotion_flag": "int8",
    # search features
    "srch_length_of_stay": "int16",
    "srch_booking_window": "int16",
    "srch_adults_count": "int8",
    "srch_children_count": "int8",
    "srch_room_count": "int8",
    "srch_saturday_night_bool": "int8",
    # nullable floats
    "srch_query_affinity_score": "float32",
    "orig_destination_distance": "float32",
    # sort flag
    "random_bool": "int8",
}

for i in range(1, 9):
    DTYPES[f"comp{i}_rate"] = "Int8"  # -1, 0, 1, null
    DTYPES[f"comp{i}_inv"] = "Int8"  # 0, 1, null
    DTYPES[f"comp{i}_rate_percent_diff"] = "float32"


In [4]:
import pandas as pd


def read_expedia_csv(path: Path) -> pd.DataFrame:
    """
    Reads the Expedia dataset CSV file into a pandas DataFrame with appropriate data types and date parsing.
    """
    cols = pd.read_csv(path, nrows=0).columns.tolist()
    dtype_for_cols = {col: dtype for col, dtype in DTYPES.items() if col in cols}
    return pd.read_csv(path, dtype=dtype_for_cols, parse_dates=["date_time"])

In [5]:
train_df = read_expedia_csv(TRAIN_PATH)
test_df = read_expedia_csv(TEST_PATH)
print(train_df.shape)
print(test_df.shape)

train_df.info(memory_usage="deep")

(4958347, 54)
(4959183, 50)
<class 'pandas.DataFrame'>
RangeIndex: 4958347 entries, 0 to 4958346
Data columns (total 54 columns):
 #   Column                       Dtype         
---  ------                       -----         
 0   srch_id                      int32         
 1   date_time                    datetime64[us]
 2   site_id                      int16         
 3   visitor_location_country_id  int16         
 4   visitor_hist_starrating      float32       
 5   visitor_hist_adr_usd         float32       
 6   prop_country_id              int16         
 7   prop_id                      int32         
 8   prop_starrating              int8          
 9   prop_review_score            float32       
 10  prop_brand_bool              int8          
 11  prop_location_score1         float32       
 12  prop_location_score2         float32       
 13  prop_log_historical_price    float32       
 14  position                     int16         
 15  price_usd                    flo

In [6]:
OUTPUT_DIR = DATA_DIR / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_df.to_parquet(OUTPUT_DIR / "training_set_VU_DM.parquet", index=False, engine="pyarrow")
test_df.to_parquet(OUTPUT_DIR / "test_set_VU_DM.parquet", index=False, engine="pyarrow")